# Seleção da montagem de *Colletotrichum abscissum* para a análise exploratória

**Objetivo:** levantar as montagens de genoma de *C. abscissum* disponíveis no NCBI, entender as diferenças entre elas
e escolher a montagem que será explorada primeiro.

**Roteiro**

1. Configuração
2. Identificador taxonômico (taxid)
3. Montagens disponíveis no NCBI
4. Tabela-resumo: tamanho, contiguidade e anotação
5. GenBank × RefSeq
6. Origem das linhagens e tecnologia de sequenciamento
7. Síntese
8. Decisão

## 1. Configuração

Importamos as bibliotecas e definimos os caminhos. Todos os caminhos partem da **raiz do projeto**
(pasta `cabs`):

```
cabs/
├── data/
│   ├── raw/          dados como vieram do NCBI
│   └── processed/    tabelas geradas pelos cadernos
├── notebooks/        cadernos
└── results/
```

In [10]:
import json
from datetime import date
from pathlib import Path
import urllib.parse
import urllib.request

import pandas as pd  # Pandas — manipulação de dados com dataframes
from Bio import Entrez  # Biopython Entrez — consultas aos bancos do NCBI

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_RAW = RAIZ / "data" / "raw"
DIR_PROC = RAIZ / "data" / "processed"
DIR_RAW.mkdir(parents=True, exist_ok=True)
DIR_PROC.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 2. Identificador taxonômico (taxid)

Cada organismo no NCBI tem um número único na base **Taxonomy**, o *taxid*. Usar o número em vez do nome evita
ambiguidades (grafias diferentes, sinônimos, nomes antigos).

Usamos o `Entrez` em duas etapas:

1. **`esearch`** — procura o nome científico na base `taxonomy` e devolve o(s) taxid(s);
2. **`efetch`** — busca o registro completo, para conferir nome, categoria (*rank*) e linhagem.

O filtro `[Scientific Name]` restringe a busca ao nome científico oficial (sem ele, a busca inclui sinônimos e nomes
populares). O NCBI pede um e-mail de contato em toda consulta pelo Entrez.

In [11]:
NOME = "Colletotrichum abscissum"

Entrez.email = "denilson.barbosa@ifsc.edu.br"
with Entrez.esearch(db="taxonomy", term=f"{NOME}[Scientific Name]") as resposta:
    busca = Entrez.read(resposta)

print("Taxids encontrados:", list(busca["IdList"]))
TAXID = str(busca["IdList"][0])

Taxids encontrados: ['1671311']


In [12]:
with Entrez.efetch(db="taxonomy", id=TAXID, retmode="xml") as resposta:
    registro = Entrez.read(resposta)[0]

print("Taxid:    ", registro["TaxId"])
print("Nome:     ", registro["ScientificName"])
print("Categoria:", registro["Rank"])
print("Linhagem: ", registro["Lineage"])

Taxid:     1671311
Nome:      Colletotrichum abscissum
Categoria: species
Linhagem:  cellular organisms; Eukaryota; Opisthokonta; Fungi; Dikarya; Ascomycota; saccharomyceta; Pezizomycotina; leotiomyceta; sordariomyceta; Sordariomycetes; Hypocreomycetidae; Glomerellales; Glomerellaceae; Colletotrichum; Colletotrichum acutatum species complex


**Resultado:** taxid **1671311**, categoria *species*.

A linhagem termina em *Colletotrichum acutatum species complex*, o complexo de espécies ao qual *C. abscissum* pertence.

## 3. Montagens disponíveis no NCBI

Consultamos a **API do NCBI Datasets**, pedindo o relatório (*dataset report*) de todas as montagens do taxid.
A resposta vem em JSON: uma lista `reports`, com um relatório por montagem, cada um com dezenas de campos aninhados.

Detalhes do código:

- **Paginação:** se houver muitas montagens, a API devolve os resultados em páginas; enquanto vier um
  `next_page_token`, pedimos a página seguinte.
- **Cópia local (cache):** o NCBI atualiza os registros com o tempo (novas montagens, novas anotações).
  Salvamos a resposta em `data/raw/`, junto com a data da consulta, e nas execuções seguintes o caderno lê esse
  arquivo — assim a análise fica reprodutível. Para atualizar, apague o arquivo e rode a célula de novo.

Equivalente na linha de comando: `datasets summary genome taxon 1671311 --as-json-lines`.

In [13]:
ARQ_RELATORIO = DIR_RAW / f"ncbi_montagens_taxon_{TAXID}.json"

def consultar_montagens(taxid):
    url_base = f"https://api.ncbi.nlm.nih.gov/datasets/v2/genome/taxon/{taxid}/dataset_report"
    relatorios, token = [], None
    while True:
        parametros = {"page_size": 100}
        if token:
            parametros["page_token"] = token
        with urllib.request.urlopen(url_base + "?" + urllib.parse.urlencode(parametros)) as resposta:
            dados = json.load(resposta)
        relatorios.extend(dados.get("reports", []))
        token = dados.get("next_page_token")
        if not token:
            return relatorios

if ARQ_RELATORIO.exists():
    cache = json.loads(ARQ_RELATORIO.read_text())
    print(f"Usando cópia local de {cache['data_consulta']}")
else:
    cache = {"data_consulta": date.today().isoformat(), "taxid": TAXID,
             "reports": consultar_montagens(TAXID)}
    ARQ_RELATORIO.write_text(json.dumps(cache, indent=1))
    print("Consulta feita e salva em", ARQ_RELATORIO.name)

relatorios = cache["reports"]
print(f"Montagens encontradas: {len(relatorios)}")

Consulta feita e salva em ncbi_montagens_taxon_1671311.json
Montagens encontradas: 4


Para ver a estrutura completa de um relatório (útil para descobrir outros campos), rode:
`print(json.dumps(relatorios[0], indent=2)[:3000])`

## 4. Tabela-resumo: tamanho, contiguidade e anotação

Extraímos de cada relatório os campos de interesse:

| Coluna | Caminho no JSON | Significado |
|---|---|---|
| `acesso` | `accession` | número de acesso (GCA = GenBank, GCF = RefSeq) |
| `linhagem` | `organism → infraspecific_names → strain` | linhagem (isolado) sequenciada |
| `submissor` | `assembly_info → submitter` | instituição que depositou a montagem |
| `tamanho_total` | `assembly_stats → total_sequence_length` | tamanho montado (pb) |
| `n_scaffolds` | `assembly_stats → number_of_scaffolds` | em quantos pedaços o genoma ficou |
| `n50_scaffold` | `assembly_stats → scaffold_n50` | contiguidade (ver abaixo) |
| `anotacao_por` | `annotation_info → provider` | quem fez a anotação dos genes |
| `genes_proteicos` | `annotation_info → stats → gene_counts → protein_coding` | genes codificadores de proteína |

**N50:** ordenando os scaffolds do maior para o menor e somando seus tamanhos, o N50 é o tamanho do scaffold em que
a soma passa da metade do genoma. Um N50 de 1 Mb significa que metade do genoma está em pedaços de 1 Mb ou maiores.
Quanto maior o N50, menos fragmentada a montagem. O número de scaffolds sozinho engana, porque uma montagem pode
ter muitos pedacinhos minúsculos e ainda assim ser boa no geral.

O encadeamento `.get(chave, {})` evita erro quando um campo não existe (uma montagem sem anotação não tem
`annotation_info`, por exemplo); nesses casos a célula fica vazia (`<NA>`).

In [14]:
linhas = []
for r in relatorios:
    info = r.get("assembly_info", {})
    stats = r.get("assembly_stats", {})
    anot = r.get("annotation_info", {})
    linhas.append({
        "acesso": r.get("accession"),
        "linhagem": r.get("organism", {}).get("infraspecific_names", {}).get("strain"),
        "submissor": info.get("submitter"),
        "tamanho_total": stats.get("total_sequence_length"),
        "n_scaffolds": stats.get("number_of_scaffolds"),
        "n50_scaffold": stats.get("scaffold_n50"),
        "anotacao_por": anot.get("provider"),
        "genes_proteicos": anot.get("stats", {}).get("gene_counts", {}).get("protein_coding"),
    })

df = pd.DataFrame(linhas)

# A API devolve alguns números como texto; "Int64" (com I maiúsculo) é o inteiro do pandas que aceita ausentes
colunas_num = ["tamanho_total", "n_scaffolds", "n50_scaffold", "genes_proteicos"]
df[colunas_num] = df[colunas_num].apply(pd.to_numeric, errors="coerce").astype("Int64")

df

,acesso,linhagem,submissor,tamanho_total,n_scaffolds,n50_scaffold,anotacao_por,genes_proteicos
0,GCA_030869225.1,IMI 504890,University of Western Brittany,53936108,560,962862,University of Western Brittany,17280
1,GCF_030869225.1,IMI 504890,University of Western Brittany,53936108,560,962862,University of Western Brittany,17223
2,GCA_047371005.1,LW-VAL-2A,University of Salamanca,53602113,50,4528207,University of Salamanca,15469
3,GCA_023376855.1,Ca142,University of Salamanca,54001896,423,321765,University of Salamanca,15499


**Leitura da tabela**

- São 4 linhas, mas **só 3 montagens diferentes**: a GCF e a GCA da linhagem IMI 504890 são a mesma montagem
  (seção 5).
- **Os tamanhos são quase iguais** (53,6 a 54,0 Mb), como se espera de linhagens da mesma espécie. As pequenas
  diferenças vêm mais da montagem (regiões repetitivas mal resolvidas, por exemplo) do que da biologia.
- **Contiguidade:** a LW-VAL-2A se destaca, com 50 scaffolds e N50 de ~4,5 Mb, contra centenas de scaffolds e N50
  entre ~0,3 e ~1 Mb das outras (a explicação está na seção 6).
- **A Ca142 já tem anotação depositada** (15.499 genes, feita em Salamanca), o mesmo número publicado no artigo.
- **As contagens de genes formam dois grupos:** ~15,5 mil (Ca142 e LW-VAL-2A, ambas anotadas em Salamanca) e ~17,2
  mil (IMI 504890, anotada em Brest). Parte dessa diferença provavelmente vem do **método de anotação**, e não da
  biologia. Anotações de pipelines diferentes não são diretamente comparáveis.

## 5. GenBank × RefSeq

O NCBI mantém dois bancos de montagens:

| | GenBank | RefSeq |
|---|---|---|
| Prefixo | **GCA_** | **GCF_** |
| Quem mantém | os autores | o NCBI |
| Conteúdo | como foi depositado | revisado e padronizado |
| Montagens por espécie | quantas forem depositadas | em geral, uma de referência |
| Anotação | opcional, no formato dos autores | obrigatória, padronizada |

- **GenBank** é o arquivo público: o que os autores depositam fica como foi enviado.
- **RefSeq** (*Reference Sequence*) é uma coleção **curada** pelo NCBI a partir do GenBank: o NCBI seleciona
  montagens, confere qualidade e contaminação, padroniza a anotação e os identificadores e publica uma cópia com
  prefixo GCF. A ideia é ter um conjunto não redundante, normalmente uma referência por espécie.

Uma montagem RefSeq é, portanto, uma **cópia curada** de uma montagem do GenBank, com o mesmo número. Abaixo,
identificamos os pares pelo número e comparamos as anotações.

**Por que a IMI 504890 virou a referência?** O NCBI considera anotação, contiguidade, completude e ausência de
contaminação. Ela tem N50 de ~963 kb (contra ~322 kb da Ca142) e foi depositada já anotada. Ser RefSeq não quer
dizer que a montagem é a melhor para todo propósito; quer dizer que é o **padrão de comparação** da espécie, com
identificadores estáveis.

In [15]:
df["banco"] = df["acesso"].str[:3].map({"GCA": "GenBank", "GCF": "RefSeq"})
df["numero"] = df["acesso"].str.extract(r"GC[AF]_(\d+)", expand=False)

pares = df[df.duplicated("numero", keep=False)].sort_values(["numero", "banco"])
pares[["acesso", "banco", "linhagem", "tamanho_total", "n_scaffolds", "n50_scaffold", "genes_proteicos"]]

,acesso,banco,linhagem,tamanho_total,n_scaffolds,n50_scaffold,genes_proteicos
0,GCA_030869225.1,GenBank,IMI 504890,53936108,560,962862,17280
1,GCF_030869225.1,RefSeq,IMI 504890,53936108,560,962862,17223


A sequência é idêntica (mesmo tamanho, scaffolds e N50); muda só a anotação: **17.280 genes no GenBank e 17.223 no
RefSeq**. A curadoria do NCBI provavelmente removeu ou corrigiu modelos que não passaram na validação (por exemplo,
CDS com códon de parada interno, genes muito curtos ou sobrepostos).

## 6. Origem das linhagens e tecnologia de sequenciamento

Agora extraímos a **tecnologia de sequenciamento**, a **data de liberação** e as informações da **amostra
biológica** (BioSample): hospedeiro, local e fonte do isolamento.

As informações do BioSample ficam numa lista de atributos (pares nome–valor), por isso usamos uma pequena função
para procurar o atributo pelo nome.

In [16]:
def atributo_biosample(relatorio, nome):
    biosample = relatorio.get("assembly_info", {}).get("biosample", {})
    if nome in biosample:
        return biosample[nome]
    for attr in biosample.get("attributes", []):
        if attr.get("name") == nome:
            return attr.get("value")
    return None

linhas = []
for r in relatorios:
    info = r.get("assembly_info", {})
    linhas.append({
        "acesso": r.get("accession"),
        "sequenciamento": info.get("sequencing_tech"),
        "liberacao": info.get("release_date"),
        "biosample": info.get("biosample", {}).get("accession"),
        "hospedeiro": atributo_biosample(r, "host"),
        "local": atributo_biosample(r, "geo_loc_name"),
        "fonte": atributo_biosample(r, "isolation_source"),
    })

df_origem = pd.DataFrame(linhas)
df_origem

,acesso,sequenciamento,liberacao,biosample,hospedeiro,local,fonte
0,GCA_030869225.1,Illumina MiSeq,2023-08-24,SAMN05949354,Citrus sinensis,USA: Florida,NaN
1,GCF_030869225.1,Illumina MiSeq,2023-08-24,SAMN05949354,Citrus sinensis,USA: Florida,NaN
2,GCA_047371005.1,PacBio RSII,2025-02-03,SAMN17142082,Citrus sinensis,"USA: Lake Wales, Polk County, Florida",flowers
3,GCA_023376855.1,Illumina MiSeq,2022-05-16,SAMN10780915,Citrus sinensis,"Brazil: Sao Paulo, Santa Cruz do Rio Pardo",flowers exhibiting petal necrosis


**Tecnologia de sequenciamento**

- **Illumina MiSeq** (IMI 504890 e Ca142): leituras **curtas** (até ~300 pb) e muito precisas. Leituras curtas não
  atravessam regiões repetitivas longas, e o genoma fica em centenas de pedaços — daí os 423 a 560 scaffolds.
- **PacBio RSII** (LW-VAL-2A): leituras **longas** (vários kb), que atravessam repetições — daí os 50 scaffolds e o
  N50 de 4,5 Mb. O RSII é uma plataforma PacBio mais antiga: as leituras (CLR) têm taxa de erro alta (~10–15%) e a
  montagem precisa ser bem "polida". Se o polimento for insuficiente, sobram pequenas inserções/deleções que quebram
  a fase de leitura dos genes. A LW-VAL-2A é a melhor em **estrutura**, mas a precisão base a base deve ser verificada
  (por exemplo, com o BUSCO).

**Data de liberação** é a data em que a montagem ficou pública, não a do sequenciamento. A amostra da IMI 504890
foi registrada anos antes da liberação da montagem (2023).

**Hospedeiro:** todas vieram de ***Citrus sinensis*** (laranja-doce), coerente com a doença causada por
*C. abscissum*: a queda prematura dos frutos cítricos (podridão floral).

**Origem:** uma linhagem do Brasil (Ca142, Santa Cruz do Rio Pardo, SP, isolada de flores com necrose nas pétalas)
e duas da Flórida, EUA (IMI 504890; LW-VAL-2A, de Lake Wales, isolada de flores).

## 7. Síntese

Juntamos as duas tabelas e ficamos com **uma linha por montagem distinta**: no par GCA/GCF, mantemos a versão
RefSeq, que tem a anotação curada.

In [17]:
sintese = (
    df.merge(df_origem, on="acesso")
      .sort_values("banco", ascending=False)        # RefSeq antes de GenBank
      .drop_duplicates("numero")                    # uma linha por montagem
      .sort_values("n50_scaffold", ascending=False)
      [["acesso", "banco", "linhagem", "local", "sequenciamento", "tamanho_total",
        "n_scaffolds", "n50_scaffold", "genes_proteicos", "anotacao_por"]]
      .reset_index(drop=True)
)
sintese

,acesso,banco,linhagem,local,sequenciamento,tamanho_total,n_scaffolds,n50_scaffold,genes_proteicos,anotacao_por
0,GCA_047371005.1,GenBank,LW-VAL-2A,"USA: Lake Wales, Polk County, Florida",PacBio RSII,53602113,50,4528207,15469,University of Salamanca
1,GCF_030869225.1,RefSeq,IMI 504890,USA: Florida,Illumina MiSeq,53936108,560,962862,17223,University of Western Brittany
2,GCA_023376855.1,GenBank,Ca142,"Brazil: Sao Paulo, Santa Cruz do Rio Pardo",Illumina MiSeq,54001896,423,321765,15499,University of Salamanca


| | Ca142 | IMI 504890 | LW-VAL-2A |
|---|---|---|---|
| Origem | Brasil (SP) | EUA (FL) | EUA (FL) |
| Tecnologia | Illumina | Illumina | PacBio |
| Ponto forte | linhagem brasileira, anotação publicada | curadoria RefSeq | contiguidade |
| Genes | 15.499 | 17.223 | 15.469 |

**Interesse para o RNAi.** A eficiência de um dsRNA depende da sequência do gene-alvo; se ela variar entre
linhagens, o silenciamento pode perder eficiência. Com uma linhagem brasileira e duas norte-americanas, será possível
procurar **alvos com sequência conservada nas três** — um indício de que o dsRNA funcionaria contra populações
diversas do patógeno. Essa comparação fica para uma etapa posterior.

## 8. Decisão

**Montagem escolhida para a exploração inicial: Ca142 — GCA_023376855.1**

Justificativa:

1. **É a linhagem brasileira**, isolada de flores de laranja-doce com sintoma de podridão floral em Santa Cruz do
   Rio Pardo (SP) — a mais próxima do contexto do projeto.
2. **Tem anotação depositada** (15.499 genes codificadores de proteína), feita com o pipeline MAKER usando dados
   transcriptômicos da própria linhagem, e descrita em artigo:
   Goulin et al. (2023), *Phytopathology* 113:104–107, doi:10.1094/PHYTO-05-22-0176-A.
3. **A patogenicidade da linhagem foi confirmada** experimentalmente pelos autores.

Limitações a ter em mente:

- A montagem é **fragmentada** (423 scaffolds, N50 ~322 kb), por ser baseada só em leituras curtas;
- A anotação vem de um pipeline diferente do usado nas outras montagens, então **contagens de genes não devem ser
  comparadas diretamente** entre elas.

A célula abaixo salva a tabela de montagens e registra a escolha.

In [18]:
ACESSO_ESCOLHIDO = "GCA_023376855.1"

ARQ_TABELA = DIR_PROC / "montagens_c_abscissum.tsv"
df.merge(df_origem, on="acesso").drop(columns="numero").to_csv(ARQ_TABELA, sep="\t", index=False)

escolhida = sintese.loc[sintese["acesso"] == ACESSO_ESCOLHIDO].iloc[0]
print("Tabela salva em:", ARQ_TABELA.relative_to(RAIZ))
print(f"\nMontagem escolhida: {escolhida['acesso']} — linhagem {escolhida['linhagem']} ({escolhida['local']})")

Tabela salva em: data/processed/montagens_c_abscissum.tsv

Montagem escolhida: GCA_023376855.1 — linhagem Ca142 (Brazil: Sao Paulo, Santa Cruz do Rio Pardo)
